# Phase 4 — Source-Aware Weak-Language Refinement

This notebook trains a Phase 4 model starting from the Phase 2 weak-pair weighted LaBSE model.

**Goal:** improve the remaining weak source languages, especially `sat`, `mni`, `brx`, and `kas`, while still keeping all 462 Indic→Indic directions in training.

Training strategy:

- Student initialization: Phase 2 best model
- Teacher / preservation model: Phase 1 balanced all-462 model
- Data: all 462 directed Indic→Indic pairs from IN22-Gen
- Sampling: weak-pair weight × source-language weight
- Loss: contrastive task loss + distillation loss
- Requested settings: `BATCH_SIZE = 512`, `EPOCHS = 5`

The notebook saves epoch checkpoints, `best_model/`, `final_model/`, and metrics.


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:

# ============================================================
# 0. Imports and environment setup
# ============================================================

import os, json, math, random, shutil, hashlib, subprocess
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device
from transformers import get_linear_schedule_with_warmup

import matplotlib.pyplot as plt

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


In [ ]:
# ============================================================
# 1. Project paths
# ============================================================
# SSH / VS Code: keep USE_GOOGLE_DRIVE = False and run this notebook from:
# ~/labse_all_pairs_indic_finetuning

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path("/content/drive/MyDrive/labse_all_pairs_indic_finetuning")
else:
    PROJECT_DIR = Path.cwd().resolve()

OUTPUT_DIR = PROJECT_DIR / "outputs"
METRICS_DIR = PROJECT_DIR / "metrics"
DATA_DIR = PROJECT_DIR / "data"

for d in [OUTPUT_DIR, METRICS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE1_RUN_NAME = "labse_all_462_directed_pairs_balanced"
PHASE2_RUN_NAME = "labse_phase2_weak_pair_weighted_from_phase1"

# Phase 4 output run name.
PHASE4_RUN_NAME = "labse_phase4_source_aware_from_phase2"

PHASE1_RUN_DIR = OUTPUT_DIR / PHASE1_RUN_NAME
PHASE2_RUN_DIR = OUTPUT_DIR / PHASE2_RUN_NAME
PHASE4_RUN_DIR = OUTPUT_DIR / PHASE4_RUN_NAME

PHASE1_BEST_MODEL_DIR = PHASE1_RUN_DIR / "best_model"
PHASE2_BEST_MODEL_DIR = PHASE2_RUN_DIR / "best_model"

PHASE4_BEST_MODEL_DIR = PHASE4_RUN_DIR / "best_model"
PHASE4_FINAL_MODEL_DIR = PHASE4_RUN_DIR / "final_model"
PHASE4_CHECKPOINT_DIR = PHASE4_RUN_DIR / "checkpoints"
PHASE4_METRICS_DIR = METRICS_DIR / PHASE4_RUN_NAME

for d in [PHASE4_RUN_DIR, PHASE4_CHECKPOINT_DIR, PHASE4_METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PHASE1_BEST_MODEL_DIR:", PHASE1_BEST_MODEL_DIR)
print("PHASE2_BEST_MODEL_DIR:", PHASE2_BEST_MODEL_DIR)
print("PHASE4_RUN_NAME:", PHASE4_RUN_NAME)
print("PHASE4_RUN_DIR:", PHASE4_RUN_DIR)
print("PHASE4_METRICS_DIR:", PHASE4_METRICS_DIR)

assert PHASE1_BEST_MODEL_DIR.exists(), f"Phase 1 best model not found: {PHASE1_BEST_MODEL_DIR}"
assert (PHASE1_BEST_MODEL_DIR / "modules.json").exists(), "Phase 1 best model is not valid"

assert PHASE2_BEST_MODEL_DIR.exists(), f"Phase 2 best model not found: {PHASE2_BEST_MODEL_DIR}"
assert (PHASE2_BEST_MODEL_DIR / "modules.json").exists(), "Phase 2 best model is not valid"


In [ ]:
# ============================================================
# 2. Hyperparameters
# ============================================================

BASE_MODEL_NAME = "sentence-transformers/LaBSE"

# Student starts from Phase 2 best model.
# Phase 4 is a source-aware refinement of the Phase 2 weak-pair weighted model.
STUDENT_INIT_MODEL_DIR = PHASE2_BEST_MODEL_DIR

# Teacher is Phase 1 balanced model.
# This preserves the globally stable all-462-pair structure.
TEACHER_MODEL_DIR = PHASE1_BEST_MODEL_DIR

MAX_SEQ_LENGTH = 128

# User-requested settings.
# If CUDA OOM occurs, reduce BATCH_SIZE to 384.
BATCH_SIZE = 512
EPOCHS = 5

# Very low LR because this is a targeted refinement from an already-good Phase 2 model.
LEARNING_RATE = 2e-7
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

EXAMPLES_PER_DIRECTED_PAIR = 500
VAL_SIZE = 0.10

# Pair weighting from Phase 2 / Phase 1 pair scores.
USE_WEIGHTED_SAMPLER = True
WEIGHT_SCORE_METRIC = "cosine_gap"
WEIGHT_ALPHA = 1.0
MIN_PAIR_WEIGHT = 1.0
MAX_PAIR_WEIGHT = 3.0
EPS = 1e-4

# Source-aware weighting for remaining weak source languages.
USE_SOURCE_WEIGHTING = True
SOURCE_WEIGHTS = {
    "sat": 5.0,  # Santali source is still very weak
    "mni": 5.0,  # Manipuri/Meitei source is still very weak
    "brx": 3.0,  # Bodo source is weak
    "kas": 3.0,  # Kashmiri source is weak
}
DEFAULT_SOURCE_WEIGHT = 1.0
MAX_COMBINED_SAMPLE_WEIGHT = 10.0

# Light preservation/distillation.
USE_DISTILLATION = True
LAMBDA_DISTILL = 0.05
MNRL_SCALE = 20.0

RESUME_FROM_LATEST_CHECKPOINT = True
CHECKPOINT_KEEP_LAST = 2

USE_AMP = torch.cuda.is_available()
EVAL_BATCH_SIZE = 256 if torch.cuda.is_available() else 64

# Best model selection: keep gap high, but penalize specificity collapse vs Phase 1 teacher.
BEST_MODEL_METRIC = "val_guarded_score"
SPECIFICITY_DROP_PENALTY = 2.0

config = {
    "phase": "phase4_source_aware_refinement",
    "run_name": PHASE4_RUN_NAME,
    "student_init_model": str(STUDENT_INIT_MODEL_DIR),
    "teacher_model": str(TEACHER_MODEL_DIR),
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_grad_norm": MAX_GRAD_NORM,
    "examples_per_directed_pair": EXAMPLES_PER_DIRECTED_PAIR,
    "val_size": VAL_SIZE,
    "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
    "weight_score_metric": WEIGHT_SCORE_METRIC,
    "weight_alpha": WEIGHT_ALPHA,
    "min_pair_weight": MIN_PAIR_WEIGHT,
    "max_pair_weight": MAX_PAIR_WEIGHT,
    "use_source_weighting": USE_SOURCE_WEIGHTING,
    "source_weights": SOURCE_WEIGHTS,
    "default_source_weight": DEFAULT_SOURCE_WEIGHT,
    "max_combined_sample_weight": MAX_COMBINED_SAMPLE_WEIGHT,
    "use_distillation": USE_DISTILLATION,
    "lambda_distill": LAMBDA_DISTILL,
    "mnrl_scale": MNRL_SCALE,
    "best_model_metric": BEST_MODEL_METRIC,
    "specificity_drop_penalty": SPECIFICITY_DROP_PENALTY,
    "resume_from_latest_checkpoint": RESUME_FROM_LATEST_CHECKPOINT,
    "checkpoint_keep_last": CHECKPOINT_KEEP_LAST,
    "seed": SEED,
}

with open(PHASE4_RUN_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


In [ ]:

# ============================================================
# 3. Language map and directed pair generation
# ============================================================

INDIC_LANGS = {
    "asm": "asm_Beng",
    "ben": "ben_Beng",
    "brx": "brx_Deva",
    "doi": "doi_Deva",
    "guj": "guj_Gujr",
    "hin": "hin_Deva",
    "kan": "kan_Knda",
    "kas": "kas_Arab",
    "gom": "gom_Deva",
    "mai": "mai_Deva",
    "mal": "mal_Mlym",
    "mni": "mni_Mtei",
    "mar": "mar_Deva",
    "npi": "npi_Deva",
    "ory": "ory_Orya",
    "pan": "pan_Guru",
    "san": "san_Deva",
    "sat": "sat_Olck",
    "snd": "snd_Deva",
    "tam": "tam_Taml",
    "tel": "tel_Telu",
    "urd": "urd_Arab",
}

LANGS = list(INDIC_LANGS.keys())
DIRECTED_PAIRS = [(src, tgt) for src in LANGS for tgt in LANGS if src != tgt]

print("Indic languages:", len(LANGS))
print("Directed Indic-Indic pairs:", len(DIRECTED_PAIRS))
assert len(DIRECTED_PAIRS) == 462


In [ ]:

# ============================================================
# 4. Load IN22-Gen and build all 462 directed pairs
# ============================================================

def load_in22_gen():
    ds = load_dataset("ai4bharat/IN22-Gen", "default", split="test")
    return ds.to_pandas()

in22_df = load_in22_gen()
print("IN22-Gen shape:", in22_df.shape)

missing_cols = [col for col in INDIC_LANGS.values() if col not in in22_df.columns]
assert not missing_cols, f"Missing language columns: {missing_cols}"

def build_all_pairs_df(base_df: pd.DataFrame, examples_per_pair: int) -> pd.DataFrame:
    rows = []
    n = min(examples_per_pair, len(base_df))

    for src, tgt in tqdm(DIRECTED_PAIRS, desc="Building all directed pairs"):
        src_col = INDIC_LANGS[src]
        tgt_col = INDIC_LANGS[tgt]

        pair_df = base_df[[src_col, tgt_col]].head(n).copy()
        pair_df = pair_df.rename(columns={src_col: "sentence1", tgt_col: "sentence2"})
        pair_df["src_lang"] = src
        pair_df["tgt_lang"] = tgt
        pair_df["direction"] = src + "→" + tgt
        pair_df["row_id"] = np.arange(n)

        pair_df = pair_df.dropna(subset=["sentence1", "sentence2"])
        pair_df["sentence1"] = pair_df["sentence1"].astype(str)
        pair_df["sentence2"] = pair_df["sentence2"].astype(str)

        rows.append(pair_df)

    return pd.concat(rows, ignore_index=True)

all_pairs_df = build_all_pairs_df(in22_df, EXAMPLES_PER_DIRECTED_PAIR)

print("Total pair rows:", len(all_pairs_df))
print("Expected max:", len(DIRECTED_PAIRS) * EXAMPLES_PER_DIRECTED_PAIR)
display(all_pairs_df.head())

all_pairs_df.to_csv(DATA_DIR / "phase4_all_462_pairs_full.csv", index=False)


In [ ]:

# ============================================================
# 5. Deterministic train/validation split per direction
# ============================================================

def stable_hash_to_float(text: str) -> float:
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF

def split_train_val(df: pd.DataFrame, val_size: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    keys = df["direction"].astype(str) + "::" + df["row_id"].astype(str)
    vals = keys.map(stable_hash_to_float)

    val_mask = vals < val_size
    train_df = df[~val_mask].reset_index(drop=True)
    val_df = df[val_mask].reset_index(drop=True)

    return train_df, val_df

train_df, val_df = split_train_val(all_pairs_df, VAL_SIZE)

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Train directions:", train_df["direction"].nunique())
print("Val directions:", val_df["direction"].nunique())

train_df.to_csv(DATA_DIR / "phase4_train_pairs_used.csv", index=False)
val_df.to_csv(DATA_DIR / "phase4_val_pairs_used.csv", index=False)


In [ ]:

# ============================================================
# 6. Evaluation helpers
# ============================================================

@torch.no_grad()
def encode_texts(model, texts, batch_size=128):
    return model.encode(
        list(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

def shifted_random_cosines(src_emb, tgt_emb, direction: str):
    n = len(src_emb)
    if n <= 1:
        return np.array([0.0])

    shift = (int(hashlib.md5(direction.encode()).hexdigest()[:8], 16) % (n - 1)) + 1
    tgt_shift = np.roll(tgt_emb, shift=shift, axis=0)
    return np.sum(src_emb * tgt_shift, axis=1)

def eval_pair_group(model, df: pd.DataFrame, batch_size=128, desc="eval"):
    rows = []

    for direction, g in tqdm(df.groupby("direction"), desc=desc):
        src_lang = g["src_lang"].iloc[0]
        tgt_lang = g["tgt_lang"].iloc[0]

        src_emb = encode_texts(model, g["sentence1"].tolist(), batch_size=batch_size)
        tgt_emb = encode_texts(model, g["sentence2"].tolist(), batch_size=batch_size)

        gold = np.sum(src_emb * tgt_emb, axis=1)
        rand = shifted_random_cosines(src_emb, tgt_emb, direction)

        mean_gold = float(np.mean(gold))
        mean_random = float(np.mean(rand))
        gap = mean_gold - mean_random

        threshold = (mean_gold + mean_random) / 2.0
        sensitivity = float(np.mean(gold >= threshold))
        specificity = float(np.mean(rand < threshold))
        balanced_accuracy = (sensitivity + specificity) / 2.0

        rows.append({
            "src_lang": src_lang,
            "tgt_lang": tgt_lang,
            "direction": direction,
            "n": len(g),
            "mean_gold_cosine": mean_gold,
            "mean_random_cosine": mean_random,
            "cosine_gap": gap,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "balanced_accuracy": balanced_accuracy,
        })

    return pd.DataFrame(rows)

def summarize_pair_metrics(pair_df: pd.DataFrame, prefix="val"):
    return {
        f"{prefix}_mean_gold_cosine": float(pair_df["mean_gold_cosine"].mean()),
        f"{prefix}_mean_random_cosine": float(pair_df["mean_random_cosine"].mean()),
        f"{prefix}_cosine_gap": float(pair_df["cosine_gap"].mean()),
        f"{prefix}_sensitivity": float(pair_df["sensitivity"].mean()),
        f"{prefix}_specificity": float(pair_df["specificity"].mean()),
        f"{prefix}_balanced_accuracy": float(pair_df["balanced_accuracy"].mean()),
    }


In [ ]:
# ============================================================
# 7. Compute/load weak-pair weights
# ============================================================
# Phase 4 still uses weak-pair weighting, but adds source-language weighting later.
# Prefer Phase 2 pair weights if they exist.
# Otherwise compute weights from Phase 1 teacher validation scores.

WEIGHTS_CANDIDATES = [
    METRICS_DIR / PHASE2_RUN_NAME / "phase2_pair_weights.csv",
    METRICS_DIR / PHASE2_RUN_NAME / "pair_weights.csv",
    PHASE2_RUN_DIR / "phase2_pair_weights.csv",
    PHASE2_RUN_DIR / "pair_weights.csv",
]

pair_weights_df = None
for p in WEIGHTS_CANDIDATES:
    if p.exists():
        print("Loading existing pair weights:", p)
        pair_weights_df = pd.read_csv(p)
        break

if pair_weights_df is None:
    print("No existing Phase 2 pair weights found.")
    print("Computing weak-pair weights from Phase 1 teacher validation scores...")

    teacher_model_for_weights = SentenceTransformer(str(TEACHER_MODEL_DIR), device=DEVICE)
    teacher_model_for_weights.max_seq_length = MAX_SEQ_LENGTH

    phase1_val_pair_scores = eval_pair_group(
        teacher_model_for_weights,
        val_df,
        batch_size=EVAL_BATCH_SIZE,
        desc="Phase 1 teacher validation pair scores",
    )

    phase1_val_pair_scores.to_csv(PHASE4_METRICS_DIR / "phase1_teacher_val_pair_scores.csv", index=False)

    pair_weights_df = phase1_val_pair_scores[[
        "src_lang", "tgt_lang", "direction", WEIGHT_SCORE_METRIC
    ]].copy()

    pair_weights_df = pair_weights_df.rename(columns={WEIGHT_SCORE_METRIC: "pair_score"})
else:
    if "pair_score" not in pair_weights_df.columns:
        if WEIGHT_SCORE_METRIC in pair_weights_df.columns:
            pair_weights_df = pair_weights_df.rename(columns={WEIGHT_SCORE_METRIC: "pair_score"})
        elif "score" in pair_weights_df.columns:
            pair_weights_df = pair_weights_df.rename(columns={"score": "pair_score"})
        else:
            raise ValueError("Existing pair weights file needs pair_score/cosine_gap/score column")

    if "direction" not in pair_weights_df.columns:
        pair_weights_df["direction"] = pair_weights_df["src_lang"].astype(str) + "→" + pair_weights_df["tgt_lang"].astype(str)

scores = pair_weights_df["pair_score"].astype(float).clip(lower=EPS)
raw_weights = (1.0 / scores) ** WEIGHT_ALPHA
raw_weights = raw_weights / np.median(raw_weights)

pair_weights_df["raw_pair_weight"] = raw_weights
pair_weights_df["pair_weight"] = np.clip(raw_weights, MIN_PAIR_WEIGHT, MAX_PAIR_WEIGHT)

pair_weights_df = pair_weights_df[[
    "src_lang", "tgt_lang", "direction", "pair_score", "raw_pair_weight", "pair_weight"
]].sort_values("pair_weight", ascending=False)

pair_weights_df.to_csv(PHASE4_METRICS_DIR / "phase4_pair_weights.csv", index=False)

print("Top weak-pair weights:")
display(pair_weights_df.head(20))
print("Pair weight summary:")
display(pair_weights_df["pair_weight"].describe())


In [ ]:
# ============================================================
# 8. Attach pair weights + source-language weights to training rows
# ============================================================

weight_map = dict(zip(pair_weights_df["direction"], pair_weights_df["pair_weight"]))

train_df["pair_weight"] = train_df["direction"].map(weight_map).fillna(1.0).astype(float)
val_df["pair_weight"] = val_df["direction"].map(weight_map).fillna(1.0).astype(float)

if USE_SOURCE_WEIGHTING:
    train_df["source_weight"] = train_df["src_lang"].map(SOURCE_WEIGHTS).fillna(DEFAULT_SOURCE_WEIGHT).astype(float)
    val_df["source_weight"] = val_df["src_lang"].map(SOURCE_WEIGHTS).fillna(DEFAULT_SOURCE_WEIGHT).astype(float)
else:
    train_df["source_weight"] = 1.0
    val_df["source_weight"] = 1.0

# Final sampler weight = weak-pair weight × source-language weight, clipped safely.
train_df["sample_weight_raw"] = train_df["pair_weight"] * train_df["source_weight"]
val_df["sample_weight_raw"] = val_df["pair_weight"] * val_df["source_weight"]

train_df["sample_weight"] = train_df["sample_weight_raw"].clip(lower=1.0, upper=MAX_COMBINED_SAMPLE_WEIGHT)
val_df["sample_weight"] = val_df["sample_weight_raw"].clip(lower=1.0, upper=MAX_COMBINED_SAMPLE_WEIGHT)

print("Train sample weight summary:")
display(train_df[["pair_weight", "source_weight", "sample_weight"]].describe())

print("Average sample weight by source language:")
display(
    train_df.groupby("src_lang")[["pair_weight", "source_weight", "sample_weight"]]
    .mean()
    .reset_index()
    .sort_values("sample_weight", ascending=False)
)

print("Top directions by final sample weight:")
display(
    train_df.groupby("direction")[["pair_weight", "source_weight", "sample_weight"]]
    .mean()
    .reset_index()
    .sort_values("sample_weight", ascending=False)
    .head(30)
)

# Save exact weighted training/validation rows for reproducibility.
train_df.to_csv(DATA_DIR / "phase4_train_pairs_source_aware_weighted.csv", index=False)
val_df.to_csv(DATA_DIR / "phase4_val_pairs_source_aware_weighted.csv", index=False)

# Save compact weight table.
source_weight_df = pd.DataFrame([
    {"src_lang": lang, "source_weight": weight}
    for lang, weight in SOURCE_WEIGHTS.items()
]).sort_values("source_weight", ascending=False)
source_weight_df.to_csv(PHASE4_METRICS_DIR / "phase4_source_weights.csv", index=False)

combined_weight_summary = (
    train_df.groupby(["src_lang", "tgt_lang", "direction"])[["pair_weight", "source_weight", "sample_weight"]]
    .mean()
    .reset_index()
    .sort_values("sample_weight", ascending=False)
)
combined_weight_summary.to_csv(PHASE4_METRICS_DIR / "phase4_combined_pair_source_weights.csv", index=False)


In [ ]:

# ============================================================
# 9. Dataset and dataloader
# ============================================================

class PairTextDataset(Dataset):
    def __init__(self, df):
        self.s1 = df["sentence1"].tolist()
        self.s2 = df["sentence2"].tolist()
        self.direction = df["direction"].tolist()

    def __len__(self):
        return len(self.s1)

    def __getitem__(self, idx):
        return self.s1[idx], self.s2[idx], self.direction[idx]

def make_dataloader(model, df, batch_size, weighted=True):
    dataset = PairTextDataset(df)

    def collate_fn(batch):
        texts1 = [x[0] for x in batch]
        texts2 = [x[1] for x in batch]
        directions = [x[2] for x in batch]

        sentence_features = [
            model.tokenize(texts1),
            model.tokenize(texts2),
        ]

        labels = torch.arange(len(batch), dtype=torch.long)
        return sentence_features, labels, directions

    if weighted:
        sample_weights = torch.tensor(df["sample_weight"].values, dtype=torch.double)
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
        )
        return DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True,
            collate_fn=collate_fn,
            num_workers=0,
        )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=collate_fn,
        num_workers=0,
    )


In [ ]:

# ============================================================
# 10. Loss functions: task + distillation
# ============================================================

def clone_feature_dict(feature_dict):
    cloned = {}
    for k, v in feature_dict.items():
        if torch.is_tensor(v):
            cloned[k] = v.detach().clone()
        else:
            cloned[k] = v
    return cloned

def mnrl_loss_from_embeddings(src_emb, tgt_emb, scale=20.0):
    src_emb = F.normalize(src_emb, p=2, dim=1)
    tgt_emb = F.normalize(tgt_emb, p=2, dim=1)

    scores = torch.matmul(src_emb, tgt_emb.T) * scale
    labels = torch.arange(scores.size(0), device=scores.device)

    return F.cross_entropy(scores, labels)

def preservation_loss(student_emb, teacher_emb):
    student_emb = F.normalize(student_emb, p=2, dim=1)
    teacher_emb = F.normalize(teacher_emb, p=2, dim=1)
    return (1.0 - F.cosine_similarity(student_emb, teacher_emb, dim=1)).mean()

def compute_task_plus_distill_loss(
    student_model,
    teacher_model,
    sentence_features,
    lambda_distill=0.05,
    mnrl_scale=20.0,
):
    src_features = sentence_features[0]
    tgt_features = sentence_features[1]

    teacher_src_features = clone_feature_dict(src_features)
    teacher_tgt_features = clone_feature_dict(tgt_features)

    student_src_emb = student_model(src_features)["sentence_embedding"]
    student_tgt_emb = student_model(tgt_features)["sentence_embedding"]

    task_loss = mnrl_loss_from_embeddings(
        student_src_emb,
        student_tgt_emb,
        scale=mnrl_scale,
    )

    with torch.no_grad():
        teacher_src_emb = teacher_model(teacher_src_features)["sentence_embedding"]
        teacher_tgt_emb = teacher_model(teacher_tgt_features)["sentence_embedding"]

    distill_src = preservation_loss(student_src_emb, teacher_src_emb)
    distill_tgt = preservation_loss(student_tgt_emb, teacher_tgt_emb)
    distill_loss = (distill_src + distill_tgt) / 2.0

    total_loss = task_loss + lambda_distill * distill_loss

    return total_loss, task_loss, distill_loss


In [ ]:

# ============================================================
# 11. Checkpoint helpers
# ============================================================

def latest_checkpoint(checkpoint_dir: Path):
    if not checkpoint_dir.exists():
        return None
    ckpts = [p for p in checkpoint_dir.glob("epoch_*") if p.is_dir()]
    if not ckpts:
        return None
    ckpts = sorted(ckpts, key=lambda p: int(p.name.split("_")[-1]))
    return ckpts[-1]

def save_checkpoint(epoch, model, optimizer, scheduler, scaler, best_score, global_step, metrics_rows):
    ckpt_dir = PHASE4_CHECKPOINT_DIR / f"epoch_{epoch:03d}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model.save(str(ckpt_dir / "model"))

    state = {
        "epoch": epoch,
        "best_score": best_score,
        "global_step": global_step,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
    }

    torch.save(state, ckpt_dir / "training_state.pt")
    pd.DataFrame(metrics_rows).to_csv(ckpt_dir / "train_metrics_until_checkpoint.csv", index=False)

    ckpts = sorted(
        [p for p in PHASE4_CHECKPOINT_DIR.glob("epoch_*") if p.is_dir()],
        key=lambda p: int(p.name.split("_")[-1])
    )

    if len(ckpts) > CHECKPOINT_KEEP_LAST:
        for old in ckpts[:-CHECKPOINT_KEEP_LAST]:
            shutil.rmtree(old, ignore_errors=True)

    print("Saved checkpoint:", ckpt_dir)


In [ ]:

# ============================================================
# 12. Load student and teacher models
# ============================================================

print("Loading student from:", STUDENT_INIT_MODEL_DIR)
student_model = SentenceTransformer(str(STUDENT_INIT_MODEL_DIR), device=DEVICE)
student_model.max_seq_length = MAX_SEQ_LENGTH

print("Loading frozen teacher from:", TEACHER_MODEL_DIR)
teacher_model = SentenceTransformer(str(TEACHER_MODEL_DIR), device=DEVICE)
teacher_model.max_seq_length = MAX_SEQ_LENGTH
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

print("Student loaded.")
print("Teacher loaded and frozen.")


In [ ]:

# ============================================================
# 13. Teacher validation baseline for guarded selection
# ============================================================

teacher_val_scores_path = PHASE4_METRICS_DIR / "teacher_val_summary.json"

if teacher_val_scores_path.exists():
    with open(teacher_val_scores_path, "r", encoding="utf-8") as f:
        teacher_val_summary = json.load(f)
    print("Loaded teacher validation summary:", teacher_val_summary)
else:
    teacher_pair_val = eval_pair_group(
        teacher_model,
        val_df,
        batch_size=EVAL_BATCH_SIZE,
        desc="Teacher validation baseline",
    )
    teacher_pair_val.to_csv(PHASE4_METRICS_DIR / "teacher_val_pair_metrics.csv", index=False)
    teacher_val_summary = summarize_pair_metrics(teacher_pair_val, prefix="teacher_val")
    with open(teacher_val_scores_path, "w", encoding="utf-8") as f:
        json.dump(teacher_val_summary, f, indent=2)

print("Teacher validation summary:")
print(json.dumps(teacher_val_summary, indent=2))

TEACHER_VAL_SPECIFICITY = teacher_val_summary["teacher_val_specificity"]


In [ ]:

# ============================================================
# 14. Training loop
# ============================================================

train_dataloader = make_dataloader(
    model=student_model,
    df=train_df,
    batch_size=BATCH_SIZE,
    weighted=USE_WEIGHTED_SAMPLER,
)

batches_per_epoch = len(train_dataloader)
total_steps = batches_per_epoch * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(student_model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

if hasattr(torch, "amp"):
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

start_epoch = 1
best_score = -float("inf")
global_step = 0
metrics_rows = []

# Full checkpoint resume
if RESUME_FROM_LATEST_CHECKPOINT:
    ckpt = latest_checkpoint(PHASE4_CHECKPOINT_DIR)
    if ckpt is not None:
        print("Resuming from:", ckpt)

        student_model = SentenceTransformer(str(ckpt / "model"), device=DEVICE)
        student_model.max_seq_length = MAX_SEQ_LENGTH

        optimizer = torch.optim.AdamW(student_model.parameters(), lr=LEARNING_RATE)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        if hasattr(torch, "amp"):
            scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        else:
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

        state = torch.load(ckpt / "training_state.pt", map_location=DEVICE)
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        if state.get("scaler_state_dict") is not None:
            scaler.load_state_dict(state["scaler_state_dict"])

        start_epoch = int(state["epoch"]) + 1
        best_score = float(state["best_score"])
        global_step = int(state["global_step"])

        mpath = ckpt / "train_metrics_until_checkpoint.csv"
        if mpath.exists():
            metrics_rows = pd.read_csv(mpath).to_dict("records")

        print("Start epoch:", start_epoch)
        print("Best score:", best_score)
        print("Global step:", global_step)

print("Training rows:", len(train_df))
print("Batches per epoch:", batches_per_epoch)
print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)
print("Start epoch:", start_epoch)

for epoch in range(start_epoch, EPOCHS + 1):
    student_model.train()

    epoch_losses = []
    epoch_task_losses = []
    epoch_distill_losses = []

    pbar = tqdm(train_dataloader, desc=f"{PHASE4_RUN_NAME} | epoch {epoch}/{EPOCHS}")

    for sentence_features, labels, directions in pbar:
        sentence_features = [batch_to_device(sf, DEVICE) for sf in sentence_features]

        optimizer.zero_grad(set_to_none=True)

        if hasattr(torch, "amp"):
            autocast_ctx = torch.amp.autocast("cuda", enabled=USE_AMP)
        else:
            autocast_ctx = torch.cuda.amp.autocast(enabled=USE_AMP)

        with autocast_ctx:
            total_loss, task_loss, distill_loss = compute_task_plus_distill_loss(
                student_model=student_model,
                teacher_model=teacher_model,
                sentence_features=sentence_features,
                lambda_distill=LAMBDA_DISTILL,
                mnrl_scale=MNRL_SCALE,
            )

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1

        loss_f = float(total_loss.detach().cpu().item())
        task_f = float(task_loss.detach().cpu().item())
        distill_f = float(distill_loss.detach().cpu().item())

        epoch_losses.append(loss_f)
        epoch_task_losses.append(task_f)
        epoch_distill_losses.append(distill_f)

        pbar.set_postfix({
            "loss": f"{loss_f:.4f}",
            "task": f"{task_f:.4f}",
            "distill": f"{distill_f:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
        })

    # Balanced validation
    student_model.eval()
    val_pair_metrics = eval_pair_group(
        student_model,
        val_df,
        batch_size=EVAL_BATCH_SIZE,
        desc=f"Validation epoch {epoch}",
    )

    val_pair_metrics.to_csv(PHASE4_METRICS_DIR / f"val_pair_metrics_epoch_{epoch:03d}.csv", index=False)

    val_summary = summarize_pair_metrics(val_pair_metrics, prefix="val")

    specificity_drop = max(0.0, TEACHER_VAL_SPECIFICITY - val_summary["val_specificity"])
    guarded_score = val_summary["val_cosine_gap"] - SPECIFICITY_DROP_PENALTY * specificity_drop

    row = {
        "epoch": epoch,
        "global_step": global_step,
        "mean_train_loss": float(np.mean(epoch_losses)),
        "mean_task_loss": float(np.mean(epoch_task_losses)),
        "mean_distill_loss": float(np.mean(epoch_distill_losses)),
        "learning_rate": float(scheduler.get_last_lr()[0]),
        "teacher_val_specificity": TEACHER_VAL_SPECIFICITY,
        "specificity_drop": specificity_drop,
        "val_guarded_score": guarded_score,
        **val_summary,
    }

    metrics_rows.append(row)
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(PHASE4_METRICS_DIR / "phase4_train_metrics.csv", index=False)
    metrics_df.to_csv(PHASE4_RUN_DIR / "train_metrics.csv", index=False)

    print("Epoch summary:")
    print(json.dumps(row, indent=2))

    current_score = row[BEST_MODEL_METRIC]
    if current_score > best_score:
        best_score = current_score
        if PHASE4_BEST_MODEL_DIR.exists():
            shutil.rmtree(PHASE4_BEST_MODEL_DIR)
        student_model.save(str(PHASE4_BEST_MODEL_DIR))
        print("New best model saved:", PHASE4_BEST_MODEL_DIR, "score:", best_score)

    save_checkpoint(
        epoch=epoch,
        model=student_model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        best_score=best_score,
        global_step=global_step,
        metrics_rows=metrics_rows,
    )

if PHASE4_FINAL_MODEL_DIR.exists():
    shutil.rmtree(PHASE4_FINAL_MODEL_DIR)
student_model.save(str(PHASE4_FINAL_MODEL_DIR))

print("Training complete.")
print("Best model:", PHASE4_BEST_MODEL_DIR)
print("Final model:", PHASE4_FINAL_MODEL_DIR)


In [ ]:

# ============================================================
# 15. Verification
# ============================================================

print("Phase 4 best model exists:", PHASE4_BEST_MODEL_DIR.exists())
print("Phase 4 best modules.json exists:", (PHASE4_BEST_MODEL_DIR / "modules.json").exists())

print("Phase 4 final model exists:", PHASE4_FINAL_MODEL_DIR.exists())
print("Phase 4 final modules.json exists:", (PHASE4_FINAL_MODEL_DIR / "modules.json").exists())

print("Metrics files:")
for p in sorted(PHASE4_METRICS_DIR.glob("*")):
    print(" -", p)


In [ ]:
# ============================================================
# 16. Optional: upload Phase 4 best_model directly to Google Drive using rclone
# No zip. Keeps folder structure.
# ============================================================

UPLOAD_PHASE4_BEST_WITH_RCLONE = False

RCLONE_REMOTE = "gdrive"
RCLONE_DRIVE_FOLDER = "labse_best_models_backup"

if UPLOAD_PHASE4_BEST_WITH_RCLONE:
    assert PHASE4_BEST_MODEL_DIR.exists()
    assert (PHASE4_BEST_MODEL_DIR / "modules.json").exists()

    drive_model_dest = f"{RCLONE_REMOTE}:{RCLONE_DRIVE_FOLDER}/phase4_source_aware_best_model/best_model"
    drive_meta_dest = f"{RCLONE_REMOTE}:{RCLONE_DRIVE_FOLDER}/phase4_source_aware_best_model/metadata"

    print("Uploading best model to:", drive_model_dest)
    subprocess.run(["rclone", "copy", str(PHASE4_BEST_MODEL_DIR), drive_model_dest, "-P"], check=True)

    print("Uploading metrics to:", drive_meta_dest)
    subprocess.run(["rclone", "copy", str(PHASE4_METRICS_DIR), f"{drive_meta_dest}/metrics", "-P"], check=True)

    print("Done uploading Phase 4 best model and metadata.")
else:
    print("Upload disabled. Set UPLOAD_PHASE4_BEST_WITH_RCLONE=True to upload.")
